# Đánh giá & Kiểm thử Thuật toán KudaNN (KNN Song song hóa trên CUDA/GPU)

Notebook này thực hiện các bước kiểm chứng và benchmark theo đúng quy trình trong báo cáo *"Paralleled K-Nearest Neighbors Algorithm"*:
1. **Kiểm tra môi trường phần cứng (GPU/CUDA) & Thư viện**.
2. **Kiểm thử tính đúng đắn (Correctness Test)** trên tập dữ liệu Iris (so sánh `SequentialKNN`, `CudaKNN` với `scikit-learn` baseline).
3. **Đo thời gian thực thi (CPU vs CUDA)** theo các kích thước mẫu tăng dần (`728`, `2912`, `3751`, `5824`, `7111`, `11648`).
4. **Tính toán & vẽ biểu đồ Hệ số tăng tốc (Acceleration)** của CUDA so với CPU.
5. **Khảo sát cấu hình số lượng Threads/Block** (`16`, `32`, `64`, `128`, `256`) trên tập dữ liệu cố định.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split

# Import package đã đóng gói trong thư mục src/
from src import KudaNNClassifier

print("CUDA Available:", cuda.is_available())
if cuda.is_available():
    device = cuda.get_current_device()
    print("Device Name:", device.name.decode('utf-8'))
    print("Compute Capability:", device.compute_capability)
    print("Multiprocessor Count:", device.MULTIPROCESSOR_COUNT)

Matplotlib is building the font cache; this may take a moment.


CUDA Available: False


## 1. Kiểm thử tính đúng đắn trên tập dữ liệu Iris
Đảm bảo cả phiên bản tuần tự (CPU) và song song (CUDA) đều cho độ chính xác tương đương chuẩn `scikit-learn`.

In [2]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

K = 3

# 1. Sklearn Baseline
clf_sklearn = KudaNNClassifier(k=K, mode='sklearn').fit(X_train, y_train)
preds_sklearn, t_sklearn = clf_sklearn.predict_with_timing(X_test)

# 2. Sequential CPU
clf_cpu = KudaNNClassifier(k=K, mode='cpu').fit(X_train, y_train)
preds_cpu, t_cpu = clf_cpu.predict_with_timing(X_test)

# 3. KudaNN (CUDA GPU)
# Chạy warm-up 1 lần để Numba biên dịch JIT kernel trước khi đo giờ thực tế
clf_cuda = KudaNNClassifier(k=K, mode='cuda', threads_per_block=256).fit(X_train, y_train)
_ = clf_cuda.predict(X_test)
preds_cuda, t_cuda = clf_cuda.predict_with_timing(X_test)

print(f"Sklearn Accuracy : {clf_sklearn.score(X_test, y_test):.6f} | Time: {t_sklearn:.6f}s")
print(f"CPU Seq Accuracy : {clf_cpu.score(X_test, y_test):.6f} | Time: {t_cpu:.6f}s")
print(f"CUDA GPU Accuracy: {clf_cuda.score(X_test, y_test):.6f} | Time: {t_cuda:.6f}s")
print("Kết quả dự đoán CPU và CUDA trùng khớp hoàn toàn:", np.array_equal(preds_cpu, preds_cuda))

RuntimeError: Không tìm thấy GPU hỗ trợ CUDA trên thiết bị này!

## 2. Benchmark CPU vs CUDA theo kích thước tập dữ liệu (Sample Size)
Sử dụng các mốc kích thước mẫu tương tự mục III.C.2 trong báo cáo: `[728, 2912, 3751, 5824, 7111, 11648]`.

In [ ]:
sample_sizes = [728, 2912, 3751, 5824, 7111, 11648]
cpu_times = []
cuda_times = []
accelerations = []

for size in sample_sizes:
    X_syn, y_syn = make_classification(
        n_samples=size, n_features=10, n_informative=6, n_classes=3, random_state=42
    )
    X_tr, X_te, y_tr, y_te = train_test_split(X_syn, y_syn, test_size=0.3, random_state=42)

    # Đo thời gian CPU
    model_cpu = KudaNNClassifier(k=5, mode='cpu').fit(X_tr, y_tr)
    _, t_c = model_cpu.predict_with_timing(X_te)

    # Đo thời gian CUDA
    model_cuda = KudaNNClassifier(k=5, mode='cuda', threads_per_block=256).fit(X_tr, y_tr)
    _, t_g = model_cuda.predict_with_timing(X_te)

    acc = t_c / t_g
    cpu_times.append(t_c)
    cuda_times.append(t_g)
    accelerations.append(acc)
    print(f"Size={size:5d} | CPU: {t_c:7.4f}s | CUDA: {t_g:7.4f}s | Acceleration: {acc:6.2f}x")

## 3. Trực quan hóa kết quả (Biểu đồ Time & Acceleration)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Thời gian thực thi CPU vs CUDA (Tương ứng Fig. 4)
x_labels = [str(s) for s in sample_sizes]
axes[0].plot(x_labels, cpu_times, marker='o', label='CPU (Sequential)', linewidth=2)
axes[0].plot(x_labels, cuda_times, marker='s', label='CUDA (GPU)', linewidth=2)
axes[0].set_title('Time taken by CPU and CUDA')
axes[0].set_xlabel('Number of nodes (Sample size)')
axes[0].set_ylabel('Time (s)')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

# Biểu đồ 2: Hệ số tăng tốc Acceleration (Tương ứng Fig. 5)
axes[1].bar(x_labels, accelerations, color='tab:blue', width=0.5, label='Acceleration')
axes[1].set_title('Acceleration of CUDA over CPU')
axes[1].set_xlabel('Number of nodes (Sample size)')
axes[1].set_ylabel('Acceleration (x)')
axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 4. Khảo sát cấu hình số Threads trên mỗi Block
Cố định tập dữ liệu 7111 mẫu, thử nghiệm với các mức `threads_per_block = [16, 32, 64, 128, 256]` (Tương ứng Fig. 6).

In [ ]:
thread_configs = [16, 32, 64, 128, 256]
thread_times = []

X_7111, y_7111 = make_classification(
    n_samples=7111, n_features=10, n_informative=6, n_classes=3, random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X_7111, y_7111, test_size=0.3, random_state=42)

for tpb in thread_configs:
    model = KudaNNClassifier(k=5, mode='cuda', threads_per_block=tpb).fit(X_tr, y_tr)
    _, elapsed = model.predict_with_timing(X_te)
    thread_times.append(elapsed)
    print(f"Threads per block = {tpb:3d} -> Time: {elapsed:.5f}s")

plt.figure(figsize=(7, 4.5))
plt.bar([str(t) for t in thread_configs], thread_times, color='tab:blue', width=0.5, label='Time taken by CUDA')
plt.title('Time taken by CUDA with different number of threads')
plt.xlabel('Number of threads')
plt.ylabel('Time (s)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()